In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AntarikshLoss(nn.Module):
    def __init__(self, lambda_vicreg=0.1, lambda_hgr=0.1):
        super().__init__()
        self.lambda_vicreg = lambda_vicreg
        self.lambda_hgr = lambda_hgr

    def off_diagonal(self, x):
        # Helper function to get off-diagonal elements of a matrix
        n, m = x.shape
        assert n == m
        return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()

    def vicreg_loss(self, x, y):
        # Flatten shape from (Batch, Hidden_Patches, Embed_Dim) to (N, Embed_Dim)
        x = x.reshape(-1, x.size(-1))
        y = y.reshape(-1, y.size(-1))
        
        # 1. Variance Loss (Prevents Collapse)
        std_x = torch.sqrt(x.var(dim=0) + 1e-04)
        std_y = torch.sqrt(y.var(dim=0) + 1e-04)
        std_loss = torch.mean(F.relu(1.0 - std_x)) + torch.mean(F.relu(1.0 - std_y))
        
        # 2. Covariance Loss (Decorrelation)
        x = x - x.mean(dim=0)
        y = y - y.mean(dim=0)
        cov_x = (x.T @ x) / (x.size(0) - 1)
        cov_y = (y.T @ y) / (y.size(0) - 1)
        cov_loss = self.off_diagonal(cov_x).pow_(2).sum() / x.size(1) + \
                   self.off_diagonal(cov_y).pow_(2).sum() / y.size(1)
                   
        return std_loss + cov_loss

    def soft_hgr_loss(self, f, g):
        # Flatten features
        f = f.reshape(-1, f.size(-1))
        g = g.reshape(-1, g.size(-1))
        
        # Zero mean
        f_centered = f - f.mean(dim=0)
        g_centered = g - g.mean(dim=0)
        
        # Covariance matrices
        cov_f = (f_centered.T @ f_centered) / (f.size(0) - 1)
        cov_g = (g_centered.T @ g_centered) / (g.size(0) - 1)
        
        # Soft-HGR: Maximize inner product, minimize trace
        correlation = torch.mean(torch.sum(f_centered * g_centered, dim=1))
        decorrelation = 0.5 * torch.trace(cov_f @ cov_g)
        
        # Negative sign because optimizer minimizes the loss, but we want to MAXIMIZE correlation
        return - (correlation - decorrelation) 

    def forward(self, predicted_sar, actual_sar):
        """
        predicted_sar: The output from CrossAttnPredictor (Phase 3)
        actual_sar: The actual features coming directly from the SAR Target Encoder
        """
        # 1. Base Prediction Error (L2 Loss)
        l2_loss = F.mse_loss(predicted_sar, actual_sar)
        
        # 2. VICReg Loss
        vicreg = self.vicreg_loss(predicted_sar, actual_sar)
        
        # 3. Soft-HGR Maximal Correlation Loss
        hgr = self.soft_hgr_loss(predicted_sar, actual_sar)
        
        # Total Objective Function
        total_loss = l2_loss + (self.lambda_vicreg * vicreg) + (self.lambda_hgr * hgr)
        
        return total_loss, l2_loss, vicreg, hgr

# --- Test the Loss Function ---
if __name__ == "__main__":
    batch_size = 32
    num_hidden = 112
    embed_dim = 192
    
    # 1. Simulate outputs
    print("Simulating Predictor Output and Actual SAR Target...")
    pred_sar = torch.randn(batch_size, num_hidden, embed_dim, requires_grad=True)
    actual_sar = torch.randn(batch_size, num_hidden, embed_dim) # Ground truth
    
    # 2. Initialize our Custom Super-Loss
    criterion = AntarikshLoss()
    
    # 3. Calculate Loss
    loss, l2, vic, hgr = criterion(pred_sar, actual_sar)
    
    print("\n--- Loss Calculation Complete ✅ ---")
    print(f"Total Loss:  {loss.item():.4f}")
    print(f"L2 Loss:     {l2.item():.4f}")
    print(f"VICReg Loss: {vic.item():.4f}")
    print(f"HGRMC Loss:  {hgr.item():.4f}")
    
    # Verify backward pass (gradients)
    loss.backward()
    print(f"\nGradient check: Success! Gradients computed for predicted SAR features.")

Simulating Predictor Output and Actual SAR Target...

--- Loss Calculation Complete ✅ ---
Total Loss:  11.5814
L2 Loss:     1.9970
VICReg Loss: 0.1185
HGRMC Loss:  95.7252

Gradient check: Success! Gradients computed for predicted SAR features.
